In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

DATA_DIR = '/content/drive/MyDrive/power_forecast/'

train_df    = pd.read_csv(DATA_DIR + 'train.csv')
test_df     = pd.read_csv(DATA_DIR + 'test.csv')
building_df = pd.read_csv(DATA_DIR + 'building_info.csv')

print('로드 완료')
print('train shape:', train_df.shape)
print('test shape :', test_df.shape)

Mounted at /content/drive
로드 완료
train shape: (204000, 10)
test shape : (16800, 7)


In [ ]:
# '-'로 표기된 결측값 0으로 변환
for col in ['태양광용량(kW)', 'ESS저장용량(kWh)', 'PCS용량(kW)']:
    building_df[col] = pd.to_numeric(building_df[col], errors='coerce').fillna(0)

# 설비 보유 여부
# 태양광이 있는 건물은 낮에 자체 발전, 순소비량이 낮게 측정됨
building_df['has_solar'] = (building_df['태양광용량(kW)'] > 0).astype(int)
building_df['has_ess']   = (building_df['ESS저장용량(kWh)'] > 0).astype(int)

# 건물유형 숫자 인코딩
le = LabelEncoder()
building_df['building_type_enc'] = le.fit_transform(building_df['건물유형'])

# IDC는 24시간 수평 패턴으로 다른 유형과 구조 자체가 다름
# 별도 플래그로 모델이 IDC를 특별하게 인식
building_df['is_IDC'] = (building_df['건물유형'] == 'IDC(전화국)').astype(int)

print('building_info 전처리 완료')
print(building_df[['건물유형', 'building_type_enc', 'is_IDC', 'has_solar', 'has_ess']].drop_duplicates())

building_info 전처리 완료
        건물유형  building_type_enc  is_IDC  has_solar  has_ess
0         호텔                  9       0          0        0
1         상용                  5       0          0        0
2         병원                  4       0          1        0
4         학교                  8       0          1        1
6       건물기타                  1       0          1        0
7         학교                  8       0          1        0
10       아파트                  6       0          0        0
12       연구소                  7       0          1        0
14       연구소                  7       0          1        1
16        병원                  4       0          0        0
17       백화점                  3       0          0        0
18       백화점                  3       0          1        0
19        상용                  5       0          1        0
29  IDC(전화국)                  0       1          0        0
32        공공                  2       0          1        0
33       백화점       

In [ ]:
# 건물번호를 키로 train/test 각 행에 건물 정보 붙이기
train_df = train_df.merge(building_df, on='건물번호', how='left')
test_df  = test_df.merge(building_df,  on='건물번호', how='left')

print('병합 후 train shape:', train_df.shape)
print('병합 후 test shape :', test_df.shape)

병합 후 train shape: (204000, 20)
병합 후 test shape : (16800, 17)


In [ ]:
def parse_datetime(df):
    df = df.copy()
    df['datetime']   = pd.to_datetime(df['일시'], format='%Y%m%d %H')
    df['month']      = df['datetime'].dt.month    # 월: 6, 7, 8
    df['day']        = df['datetime'].dt.day      # 일: 1~31
    df['hour']       = df['datetime'].dt.hour     # 시간: 0~23
    df['weekday']    = df['datetime'].dt.weekday  # 요일: 0(월)~6(일)

    # 연구소/학교는 주말에 감소, 아파트/호텔은 주말에 증가
    # 주말 여부를 명시적으로 feature로 추가
    df['is_weekend'] = (df['weekday'] >= 5).astype(int)
    return df

train_df = parse_datetime(train_df)
test_df  = parse_datetime(test_df)

print('시간 파싱 완료')
display(train_df[['일시', 'month', 'day', 'hour', 'weekday', 'is_weekend']].head())

시간 파싱 완료


,일시,month,day,hour,weekday,is_weekend
0,20240601 00,6,1,0,5,1
1,20240601 01,6,1,1,5,1
2,20240601 02,6,1,2,5,1
3,20240601 03,6,1,3,5,1
4,20240601 04,6,1,4,5,1


In [ ]:
# 건물 유형마다 피크 시간대가 다름
# sin/cos 변환으로 시간 원형 구조 표현

for df in [train_df, test_df]:
    df['hour_sin']    = np.sin(2 * np.pi * df['hour']    / 24)
    df['hour_cos']    = np.cos(2 * np.pi * df['hour']    / 24)
    df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
    df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)

print('주기성 인코딩 완료')
print('0시: hour_sin: {:.3f}, hour_cos: {:.3f}'.format(
    np.sin(2*np.pi*0/24), np.cos(2*np.pi*0/24)))
print('23시: hour_sin: {:.3f}, hour_cos: {:.3f}'.format(
    np.sin(2*np.pi*23/24), np.cos(2*np.pi*23/24)))
print('(두 값이 비슷: 0시와 23시가 가깝다는 의미)')

주기성 인코딩 완료
0시: hour_sin: 0.000, hour_cos: 1.000
23시: hour_sin: -0.259, hour_cos: 0.966
(두 값이 비슷: 0시와 23시가 가깝다는 의미)


In [ ]:
# 불쾌지수(THI): 0.81 * 기온 + 0.01 * 습도 * (0.99 * 기온 - 14.99) + 46.3
# 기온 단독보다 기온+습도 조합이 냉방 수요를 더 잘 설명
# 같은 30°C라도 습도 40% vs 90%는 냉방 수요가 전혀 다름

# 냉방도시(CDH): max(기온 - 26, 0)
# 기온 구간별 분석에서 26°C 이상부터 전력 증가 속도가 급격히 빨라짐
# 26°C를 냉방 기준 온도로 설정

COOLING_BASE = 26

for df in [train_df, test_df]:
    T  = df['기온(°C)']
    RH = df['습도(%)']

    df['THI'] = 0.81*T + 0.01*RH*(0.99*T - 14.99) + 46.3
    df['CDH'] = (T - COOLING_BASE).clip(lower=0)

print('기상 파생 변수 완료')
display(train_df[['기온(°C)', '습도(%)', 'THI', 'CDH']].describe().round(2))

기상 파생 변수 완료


,기온(°C),습도(%),THI,CDH
count,204000.00,204000.00,204000.00,204000.00
mean,26.10,75.21,75.31,1.66
std,4.05,16.38,5.67,2.28
min,8.40,0.00,46.90,0.00
25%,23.50,64.00,71.92,0.00
50%,26.30,78.00,76.17,0.30
75%,28.80,88.00,79.43,2.80
max,38.70,100.00,89.90,12.70


In [ ]:
# 건물 유형 간 최대 9.3배 전력 차이
# "이 건물이 이 시간대에 평소 얼마나 쓰는가"를 직접 알려줌
# train에서만 계산 후 test에 붙임

# 건물 × 시간대 평균
building_hour_mean = (
    train_df.groupby(['건물번호', 'hour'])['전력소비량(kWh)']
    .mean().reset_index()
    .rename(columns={'전력소비량(kWh)': 'building_hour_mean'})
)

# 건물 × 요일 평균
building_weekday_mean = (
    train_df.groupby(['건물번호', 'weekday'])['전력소비량(kWh)']
    .mean().reset_index()
    .rename(columns={'전력소비량(kWh)': 'building_weekday_mean'})
)

# 건물 전체 평균 (규모 반영)
building_mean = (
    train_df.groupby('건물번호')['전력소비량(kWh)']
    .mean().reset_index()
    .rename(columns={'전력소비량(kWh)': 'building_mean'})
)

train_df = train_df.merge(building_hour_mean,    on=['건물번호', 'hour'],    how='left')
train_df = train_df.merge(building_weekday_mean, on=['건물번호', 'weekday'], how='left')
train_df = train_df.merge(building_mean,         on='건물번호',              how='left')

test_df  = test_df.merge(building_hour_mean,    on=['건물번호', 'hour'],    how='left')
test_df  = test_df.merge(building_weekday_mean, on=['건물번호', 'weekday'], how='left')
test_df  = test_df.merge(building_mean,         on='건물번호',              how='left')

print('건물별 통계 feature 완료')
display(train_df[['건물번호', 'hour', 'building_hour_mean', 'building_mean']].head())

건물별 통계 feature 완료


,건물번호,hour,building_hour_mean,building_mean
0,1,0,5258.268000,5340.317485
1,1,1,5019.340941,5340.317485
2,1,2,4810.891412,5340.317485
3,1,3,4529.493529,5340.317485
4,1,4,4304.295882,5340.317485


In [ ]:
FEATURES = [
    # 건물 정보
    '건물번호', 'building_type_enc',
    '연면적(m2)', '냉방면적(m2)', '태양광용량(kW)', 'ESS저장용량(kWh)', 'PCS용량(kW)',
    'has_solar', 'has_ess',
    'is_IDC',                    # EDA 신규 추가

    # 시간
    'month', 'day', 'hour', 'weekday', 'is_weekend',
    'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos',

    # 기상
    '기온(°C)', '강수량(mm)', '풍속(m/s)', '습도(%)', 'THI', 'CDH',

    # 건물별 통계
    'building_hour_mean', 'building_weekday_mean', 'building_mean',
]

# 일조/일사는 train에만 있고 test에는 없음
TRAIN_FEATURES = FEATURES + ['일조(hr)', '일사(MJ/m2)']
TARGET = '전력소비량(kWh)'

train_x = train_df[TRAIN_FEATURES]
train_y = train_df[TARGET]
test_x  = test_df[FEATURES]

print(f'=== Feature Engineering 완료 ===')
print(f'train feature 수 : {len(TRAIN_FEATURES)}개')
print(f'train_x shape    : {train_x.shape}')
print(f'test_x shape     : {test_x.shape}')
print(f'결측치 (train_x) : {train_x.isnull().sum().sum()}개')
print(f'결측치 (test_x)  : {test_x.isnull().sum().sum()}개')

=== Feature Engineering 완료 ===
train feature 수 : 30개
train_x shape    : (204000, 30)
test_x shape     : (16800, 28)
결측치 (train_x) : 0개
결측치 (test_x)  : 0개


In [ ]:
# 가공된 데이터 드라이브에 저장
train_df.to_csv(DATA_DIR + 'train_fe.csv', index=False)
test_df.to_csv(DATA_DIR + 'test_fe.csv',   index=False)

print('저장 완료')
print(f'train_fe.csv: {train_df.shape}')
print(f'test_fe.csv : {test_df.shape}')

저장 완료
train_fe.csv: (204000, 35)
test_fe.csv : (16800, 32)
